In [3]:
!pip -q install gradio transformers torch librosa soundfile gTTS SpeechRecognition pydub nltk

In [4]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# ============================================================
# VOCASENSE AI
# Intelligent Text & Speech Analysis with Adaptive TTS
# ============================================================

import gradio as gr
import re
import os
import tempfile
import librosa
import speech_recognition as sr

from collections import Counter
from pydub import AudioSegment
from gtts import gTTS
from transformers import pipeline


# ============================================================
# AI SENTIMENT MODEL
# ============================================================

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)


# ============================================================
# AUDIO CONVERSION
# ============================================================

def convert_to_wav(audio_file):

    audio = AudioSegment.from_file(audio_file)

    # Convert to standard format
    audio = audio.set_channels(1)
    audio = audio.set_frame_rate(16000)
    audio = audio.set_sample_width(2)

    wav_file = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ).name

    audio.export(
        wav_file,
        format="wav"
    )

    return wav_file


# ============================================================
# TEXT ANALYSIS
# ============================================================

def analyze_text(text):

    if not text or not text.strip():

        return (
            "Please enter some text.",
            "",
            ""
        )

    text = text.strip()

    # Words
    words = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    # Sentences
    sentences = re.split(
        r"[.!?]+",
        text
    )

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    word_count = len(words)

    character_count = len(text)

    sentence_count = len(sentences)

    # Word frequency
    frequency = Counter(
        word.lower()
        for word in words
    )

    common_words = frequency.most_common(10)

    keywords = ", ".join(
        word
        for word, count in common_words
    )

    # Sentiment
    try:

        result = sentiment_model(
            text[:512]
        )[0]

        sentiment = (
            f"{result['label']} "
            f"({result['score']:.2%})"
        )

    except Exception:

        sentiment = "Unable to determine sentiment."

    # Statistics
    statistics = f"""
### 📊 Text Statistics

**Words:** {word_count}

**Characters:** {character_count}

**Sentences:** {sentence_count}

**Average words/sentence:** {
    round(
        word_count / max(sentence_count, 1),
        2
    )
}
"""

    # Keywords
    keyword_result = f"""
### 🔑 Important Words

{keywords}
"""

    # Sentiment
    sentiment_result = f"""
### 😊 Sentiment Analysis

**Result:** {sentiment}
"""

    return (
        statistics,
        keyword_result,
        sentiment_result
    )


# ============================================================
# SPEECH TO TEXT
# ============================================================

def speech_to_text(audio):

    if audio is None:

        return "Please record or upload speech."

    wav_file = None

    try:

        # Convert browser audio to WAV
        wav_file = convert_to_wav(audio)

        recognizer = sr.Recognizer()

        with sr.AudioFile(wav_file) as source:

            # Reduce background noise
            recognizer.adjust_for_ambient_noise(
                source,
                duration=0.5
            )

            audio_data = recognizer.record(
                source
            )

        # Google Speech Recognition
        text = recognizer.recognize_google(
            audio_data,
            language="en-US"
        )

        return text

    except sr.UnknownValueError:

        return (
            "Could not understand the speech. "
            "Please speak clearly and try again."
        )

    except sr.RequestError:

        return (
            "Speech recognition service is unavailable. "
            "Please check your internet connection."
        )

    except Exception as e:

        return f"Speech processing error: {str(e)}"

    finally:

        if (
            wav_file
            and os.path.exists(wav_file)
        ):

            os.remove(wav_file)


# ============================================================
# SPEECH ANALYSIS
# ============================================================

def analyze_speech(audio):

    if audio is None:

        return (
            """
### 🎤 Speech Analysis

Please record or upload an audio file.
""",
            ""
        )

    wav_file = None

    try:

        # Convert audio
        wav_file = convert_to_wav(audio)

        # Load audio
        y, sr_rate = librosa.load(
            wav_file,
            sr=None
        )

        # Duration
        duration = librosa.get_duration(
            y=y,
            sr=sr_rate
        )

        # Voice energy
        rms = librosa.feature.rms(
            y=y
        )

        volume = float(
            rms.mean()
        )

        # Speech recognition
        recognizer = sr.Recognizer()

        with sr.AudioFile(wav_file) as source:

            recognizer.adjust_for_ambient_noise(
                source,
                duration=0.5
            )

            audio_data = recognizer.record(
                source
            )

        try:

            speech_text = recognizer.recognize_google(
                audio_data,
                language="en-US"
            )

        except sr.UnknownValueError:

            speech_text = (
                "Could not understand the speech."
            )

        except sr.RequestError:

            speech_text = (
                "Speech recognition service "
                "is temporarily unavailable."
            )

        # Word count
        words = speech_text.split()

        word_count = len(words)

        # Speaking rate
        if duration > 0 and word_count > 0:

            speaking_rate = (
                word_count / duration
            ) * 60

        else:

            speaking_rate = 0

        # Speaking speed
        if speaking_rate < 100:

            speed_status = "Slow"

        elif speaking_rate <= 160:

            speed_status = "Normal"

        else:

            speed_status = "Fast"

        # Final result
        result = f"""
### 🎤 Speech Analysis

**Duration:** {duration:.2f} seconds

**Detected Words:** {word_count}

**Speaking Rate:** {speaking_rate:.2f} words/minute

**Speaking Speed:** {speed_status}

**Voice Energy:** {volume:.4f}

---

### 📝 Transcription

{speech_text}
"""

        return (
            result,
            speech_text
        )

    except Exception as e:

        return (
            f"""
### ❌ Speech Analysis Error

{str(e)}
""",
            ""
        )

    finally:

        if (
            wav_file
            and os.path.exists(wav_file)
        ):

            os.remove(wav_file)


# ============================================================
# TEXT TO SPEECH
# ============================================================

def text_to_speech(text):

    if not text or not text.strip():

        return None

    try:

        filename = (
            "/content/"
            "vocasense_output.mp3"
        )

        tts = gTTS(
            text=text,
            lang="en",
            slow=False
        )

        tts.save(filename)

        return filename

    except Exception as e:

        print(
            "TTS Error:",
            e
        )

        return None


# ============================================================
# TEXT PROCESSING
# ============================================================

def process_text(text):

    statistics, keywords, sentiment = analyze_text(
        text
    )

    return (
        statistics,
        keywords,
        sentiment
    )


# ============================================================
# PROFESSIONAL CSS
# ============================================================

custom_css = """

body {
    background:
        linear-gradient(
            135deg,
            #eef2ff,
            #f5f3ff
        );
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto;
}

.title {
    text-align: center;
    font-size: 42px;
    font-weight: 800;
    color: #4f46e5;
    margin-bottom: 5px;
}

.subtitle {
    text-align: center;
    font-size: 18px;
    color: #555;
    margin-bottom: 25px;
}

button {
    border-radius: 10px !important;
}

textarea,
input {
    border-radius: 12px !important;
}

"""


# ============================================================
# MAIN APPLICATION
# ============================================================

with gr.Blocks(
    css=custom_css,
    title="VocaSense AI"
) as app:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.Markdown(
        """
        <div class="title">
        🎙️ VocaSense AI
        </div>

        <div class="subtitle">
        Intelligent Text & Speech Analysis
        with Adaptive Text-to-Speech
        </div>
        """
    )


    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():


        # ====================================================
        # TEXT ANALYSIS
        # ====================================================

        with gr.Tab(
            "📝 Text Analysis"
        ):

            text_input = gr.Textbox(
                label="Enter Your Text",
                placeholder=(
                    "Type or paste your text here..."
                ),
                lines=8
            )

            analyze_button = gr.Button(
                "🔍 Analyze Text",
                variant="primary"
            )

            with gr.Row():

                statistics_output = gr.Markdown(
                    label="Statistics"
                )

                keyword_output = gr.Markdown(
                    label="Keywords"
                )

            sentiment_output = gr.Markdown(
                label="Sentiment"
            )

            analyze_button.click(
                process_text,
                inputs=text_input,
                outputs=[
                    statistics_output,
                    keyword_output,
                    sentiment_output
                ]
            )


        # ====================================================
        # SPEECH ANALYSIS
        # ====================================================

        with gr.Tab(
            "🎤 Speech Analysis"
        ):

            audio_input = gr.Audio(
                sources=[
                    "microphone",
                    "upload"
                ],
                type="filepath",
                label="Upload or Record Speech"
            )

            speech_button = gr.Button(
                "🎧 Analyze Speech",
                variant="primary"
            )

            speech_output = gr.Markdown()

            transcription_output = gr.Textbox(
                label="Speech Transcription",
                lines=5
            )

            speech_button.click(
                analyze_speech,
                inputs=audio_input,
                outputs=[
                    speech_output,
                    transcription_output
                ]
            )


        # ====================================================
        # TEXT TO SPEECH
        # ====================================================

        with gr.Tab(
            "🔊 Text-to-Speech"
        ):

            tts_input = gr.Textbox(
                label="Enter Text",
                placeholder=(
                    "Enter text that you want "
                    "to convert into speech..."
                ),
                lines=8
            )

            tts_button = gr.Button(
                "🔊 Generate Speech",
                variant="primary"
            )

            audio_output = gr.Audio(
                label="Generated Voice"
            )

            tts_button.click(
                text_to_speech,
                inputs=tts_input,
                outputs=audio_output
            )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.Markdown(
        """
        ---

        ### 🚀 VocaSense AI

        **Text Analysis • Speech Analysis •
        Sentiment Detection • Speech-to-Text •
        Text-to-Speech**

        Developed as an AI-powered
        Text and Speech Intelligence System.
        """
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=True
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_6722/3329596784.py:490: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a6c29bdc53848ec6fe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
